# Attention-probing arm only

Standalone version of the `c1_declare.ipynb` ablation, for when a session cannot hold the
whole notebook. It runs only `read`, `attention` and `attention_late` - it does **not** re-run
`generate`, which is the expensive mode (~95 min at 1.5B) and whose results are already
recorded.

## Seeds

The crossover this notebook measures - `read` ahead at 0.5B, `attention_late` nominally ahead
at 1.5B - was a single run at a single layout draw, and the 1.5B margin was 0.391 vs 0.372,
well inside noise. `SEEDS` now runs the whole grid at several seeds and
`eval/declare_report.py` reports mean ± sd with a paired t across them.

What a seed changes is the **layout**: the gold-region permutation and the content shuffle.
The probe set is fixed by the data file, so this measures robustness to layout, not to data
sampling. Say that rather than implying full resampling.

`generate` is not re-run at the new seeds. The comparison that matters is `read` against
`attention_late`, which does not involve it, and the report warns that the seed counts are
unbalanced.

## Layout guard

**The one thing that can invalidate this run** is landing on different layouts from the run it
must be compared against. Layouts depend on the data, `n_eval` and the seed. `--gold-ref`
points at a per-seed file holding the gold-region distribution: the first run at a seed writes
it, and every later run at that seed refuses to start unless it matches. `GOLD_S0` seeds that
file for seed 0 with the distribution from the recorded `generate`/`read` run, so seed 0 stays
pinned to it.

Two further guards run automatically:

- `run.py` compares the probe attention implementation against SDPA on a left-padded batch and
  aborts if log-probabilities diverge. An unregistered mask implementation silently dropped the
  padding mask once already, which corrupted `read`.
- `read` is recomputed here. **At seed 0 only**, it must reproduce **0.333** at 0.5B and
  **0.372** at 1.5B. If it does not, stop and do not use the attention numbers. At other seeds
  it will differ; that spread is the thing being measured.

Restore the previous `runs_declare.zip` in the restore cell so `generate` merges into the final
report. Without it the report shows only the three modes this notebook produces.


In [ ]:
import glob, os, subprocess, sys, zipfile

GIT_URL = 'https://github.com/rajul-kk/context-to-weights.git'
REPO = '/kaggle/working/myrios'
MARKER = 'baselines/cascading.py'
FRESH = True
REQUIRE = [('declare/run.py', 'gold-ref'), ('eval/declare_report.py', 'paired')]

def looks_like_source(d):
    return os.path.exists(os.path.join(d, MARKER))

if FRESH and GIT_URL and os.path.exists(REPO):
    subprocess.run(['rm', '-rf', REPO], check=True)
    print(f'removed {REPO} so the clone is current')

if not looks_like_source(REPO) and GIT_URL:
    r = subprocess.run(['git', 'clone', '--depth', '1', GIT_URL, REPO],
                       capture_output=True, text=True)
    print(r.stdout, r.stderr)

if not looks_like_source(REPO):
    print('no clone; falling back to attached inputs')
    src = None
    for d in sorted(glob.glob('/kaggle/input/*')):
        if looks_like_source(d):
            src = d
            break
        for z in sorted(glob.glob(os.path.join(d, '*.zip'))):
            os.makedirs(REPO, exist_ok=True)
            zipfile.ZipFile(z).extractall(REPO)
            if looks_like_source(REPO):
                src = REPO
                break
        if src:
            break
    if src and src != REPO:
        subprocess.run(['cp', '-r', src, REPO], check=True)

assert looks_like_source(REPO), (
    'No source found. Either (a) set GIT_URL above, or (b) run scripts/package_source.py '
    'locally, upload the zip as a Kaggle Dataset, and attach it via Add Input. '
    f'Searched /kaggle/input/*, saw: {sorted(glob.glob("/kaggle/input/*"))}')

os.chdir(REPO)
sys.path.insert(0, REPO)

_h = subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True, cwd=REPO)
print('source commit:', _h.stdout.strip() or 'unknown (not a git checkout)')

_stale = [f for f, token in REQUIRE
          if not os.path.exists(f) or token not in open(f, encoding='utf-8').read()]
assert not _stale, (
    f'This checkout is older than the seed support these cells need: {_stale}. '
    'It came from an attached dataset or a leftover /kaggle/working/myrios rather than a '
    'fresh clone. Detach any source dataset under Add Input, keep only the runs_declare.zip '
    'dataset, and re-run this cell with FRESH = True.')
print('source is current for the seed grid')

subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'peft', 'accelerate', 'datasets'])

try:
    import importlib.metadata as _md
    _v = _md.version('torchao')
    if tuple(int(x) for x in _v.split('.')[:2]) < (0, 16):
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'uninstall', '-y', 'torchao'])
        print(f'removed incompatible torchao {_v}')
except Exception:
    pass

exec(open('notebooks/_runner.py').read())
print('cwd', os.getcwd())
print('files', sorted(os.listdir('.'))[:10])
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    print()
    print('=' * 68)
    print('NO GPU. Kaggle installed the CPU build of torch, so this session')
    print('has no accelerator attached. Everything below will be far too slow.')
    print()
    print('Fix: right panel -> Session options -> Accelerator -> GPU T4 x2,')
    print('then Run All again. The image swaps to a CUDA torch build on restart.')
    print('=' * 68)
else:
    print('gpu', torch.cuda.get_device_name(0),
          f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


In [ ]:
# re-run this cell (only) if the kernel restarts; it re-derives everything
import os, sys
REPO = '/kaggle/working/myrios'
os.chdir(REPO); sys.path.insert(0, REPO)
if 'run' not in dir():
    exec(open('notebooks/_runner.py').read())

CFG = 'configs/kaggle_declare.yaml'
RUNS = '/kaggle/working/artifacts/runs_declare'
DATA = 'artifacts/data/hotpotqa_spread'
N_EVAL = 96
N_PROBES = 0             # 0 = every probe
MODES = 'read attention attention_late'
MODELS = ['Qwen/Qwen2.5-0.5B-Instruct', 'Qwen/Qwen2.5-1.5B-Instruct']
SEEDS = [0, 1, 2]
GOLD_S0 = {"0": 51, "1": 51, "2": 38, "3": 57, "4": 38, "5": 51, "6": 49, "7": 49}

from common.io import load_config, write_json
_c = load_config(CFG)
assert _c['run_root'] == RUNS, f"run_root mismatch: {_c['run_root']} != {RUNS}"
assert _c['data']['dir'] == DATA, f"data.dir mismatch: {_c['data']['dir']} != {DATA}"

os.makedirs(RUNS, exist_ok=True)
if not os.path.exists(f'{RUNS}/gold_s0.json'):
    write_json(f'{RUNS}/gold_s0.json', GOLD_S0)

def out_dir(slug, seed):
    return f'{RUNS}/{slug}' if seed == 0 else f'{RUNS}/{slug}_s{seed}'

print(f'modes: {MODES}')
print(f'seeds: {SEEDS}  ({len(SEEDS) * len(MODELS)} runs)')
print(f'eval trajectories {N_EVAL}, regions {_c["declare"]["n_regions"]}')
print(f'seed 0 layouts must match gold {GOLD_S0}')


In [ ]:
# Restore the runs_declare.zip from the generate/read session so its summaries
# merge into the final report. Attach it as an input dataset and set ARCHIVE.
ARCHIVE = '/kaggle/input/CHANGE-ME/runs_declare.zip'
import os
if os.path.exists(ARCHIVE):
    run(f"python scripts/kaggle_sync.py restore --archive {ARCHIVE} --run-root {RUNS}")
else:
    print(f'no archive at {ARCHIVE}')
    print('the run still works, but the report will omit generate unless you restore it')


In [ ]:
run(f"python data/load_hotpotqa.py --n-train 8 --n-eval {N_EVAL} --per-trajectory 4 --early-frac 1.0 --out {DATA}")


## The ablation

`attention` reads the attention mass the final query position puts on each region and takes the
argmax. `attention_late` uses only the second half of the layers. Same layouts, same probes and
same model as `read`, so the three elicitations are directly comparable.

Budget roughly 25 min at 0.5B and 40 min at 1.5B, so about **65 min per seed** and **3.3 h for
the three-seed grid**. That does not fit one session: run one seed per session, keep the
`runs_declare.zip` and restore it into the next.


In [ ]:
lim = f'--limit {N_PROBES}' if N_PROBES else ''
total = len(SEEDS) * len(MODELS)
i = 0
for seed in SEEDS:
    for m in MODELS:
        i += 1
        slug = m.split('/')[-1]
        print('=' * 70)
        print(f'[{i}/{total}] {slug} seed {seed}', flush=True)
        run(f"python declare/run.py --config {CFG} --modes {MODES} {lim} "
            f"--out {out_dir(slug, seed)} --gold-ref {RUNS}/gold_s{seed}.json "
            f"--set model.base={m} seed={seed}")
        run(f"python scripts/kaggle_sync.py save --run-root {RUNS} "
            f"--archive /kaggle/working/runs_declare.zip")


In [ ]:
run(f"python eval/declare_report.py --run-root {RUNS} --out docs/results_declare.md")
from IPython.display import Image, Markdown, display
import os
fig = f'{RUNS}/figures/declare.png'
if os.path.exists(fig):
    display(Image(fig))
if os.path.exists('docs/results_declare.md'):
    display(Markdown(open('docs/results_declare.md').read()))
run(f"python scripts/kaggle_sync.py save --run-root {RUNS} --archive /kaggle/working/runs_declare.zip")
print()
print('=' * 64)
print('DOWNLOAD /kaggle/working/runs_declare.zip (right panel -> Output)')
print('=' * 64)
